# 03 · Training
Three-phase curriculum: cell-level → aggregator → end-to-end.

In [ ]:
import sys, torch
sys.path.insert(0, '../src')
from model import MultiSmokeCancerNet
from train import Trainer, CellLevelDataset, SubjectLevelDataset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

CFG = '../configs/default.yaml'
model   = MultiSmokeCancerNet.from_config(CFG)
trainer = Trainer.from_config(model, CFG, device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Load preprocessed data

In [ ]:
from pathlib import Path
import numpy as np

processed = Path('../data/processed')
if processed.exists() and (processed/'gene_matrix.npy').exists():
    cell_ds = CellLevelDataset.from_dir(processed)
    print(f'Loaded {len(cell_ds):,} cells')
else:
    print('No real data — run 02_preprocessing.ipynb first')
    print('Using synthetic data for demo...')
    from constants import N_SMOKE_CLASSES, N_CELL_TYPES
    import random
    cell_ds = CellLevelDataset(
        gene_matrix       = np.random.randn(2000, 2000).astype('float32'),
        smoke_labels      = np.random.randint(0, N_SMOKE_CLASSES, 2000),
        malignancy_labels = np.random.randint(0, 2, 2000).astype('float32'),
        cell_type_ids     = np.random.randint(0, N_CELL_TYPES, 2000),
    )
    def _bag(n):
        return {'gene_matrix': np.random.randn(n,2000).astype('float32'),
                'cell_type_ids': np.random.randint(0,N_CELL_TYPES,n),
                'smoke_labels': np.random.randint(0,N_SMOKE_CLASSES,n),
                'malig_labels': np.random.randint(0,2,n).astype('float32'),
                'cancer_label': random.randint(0,1)}
    subject_ds = SubjectLevelDataset([
        {'subject_id':f's{i}',**_bag(random.randint(40,100))} for i in range(30)
    ])

## Phase 1 — Cell-level pre-training

In [ ]:
r1 = trainer.phase1(cell_ds)
print(f"Best smoke_acc: {r1['best_smoke_acc']:.3f}")

## Phase 2 — Aggregator training

In [ ]:
r2 = trainer.phase2(subject_ds)
print(f"Best AUC: {r2['best_auc']:.3f}")

## Phase 3 — End-to-end fine-tuning

In [ ]:
r3 = trainer.phase3(cell_ds, subject_ds)
print(f"Best AUC: {r3['best_auc']:.3f}")

## Plot training curves

In [ ]:
from visualize import plot_training_history
plot_training_history(out_dir='../checkpoints/plots')
from IPython.display import Image
Image('../checkpoints/plots/training_history.png')